In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
def run_sql_script(sql_script: str):
    # Create in-memory SQLite database
    engine = create_engine("sqlite:///:memory:")

    # Replace TRUNCATE with DELETE (SQLite doesn't support TRUNCATE)
    lines = []
    for line in sql_script.strip().splitlines():
        line = line.strip()
        if line.lower().startswith("truncate table"):
            table_name = line.split()[2]
            lines.append(f"DELETE FROM {table_name}")
        else:
            lines.append(line)

    cleaned_script = "\n".join(lines)

    with engine.connect() as conn:
        # Execute full script
        for statement in cleaned_script.split("\n"):
            if statement.strip():
                conn.execute(text(statement))
        conn.commit()

    return engine

### 175. Combine Two Tables

In [3]:
sql_script = """
Create table If Not Exists Person (personId int, firstName varchar(255), lastName varchar(255))
Create table If Not Exists Address (addressId int, personId int, city varchar(255), state varchar(255))
Truncate table Person
insert into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen')
insert into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob')
Truncate table Address
insert into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York')
insert into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California')
"""

engine = run_sql_script(sql_script)

In [4]:
sql = """
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a
ON p.personId = a.personId;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,firstName,lastName,city,state
0,Allen,Wang,NaN,NaN
1,Bob,Alice,New York City,New York


In [5]:
data = [[1, 'Wang', 'Allen'], [2, 'Alice', 'Bob']]
person = pd.DataFrame(data, columns=['personId', 'firstName', 'lastName']).astype({'personId':'Int64', 'firstName':'object', 'lastName':'object'})
data = [[1, 2, 'New York City', 'New York'], [2, 3, 'Leetcode', 'California']]
address = pd.DataFrame(data, columns=['addressId', 'personId', 'city', 'state']).astype({'addressId':'Int64', 'personId':'Int64', 'city':'object', 'state':'object'})

In [6]:
person.merge(address, on='personId', how='left')[['firstName', 'lastName', 'city', 'state']]

,firstName,lastName,city,state
0,Wang,Allen,NaN,NaN
1,Alice,Bob,New York City,New York


### 176. Second Highest Salary

In [7]:
sql_script = """
Create table If Not Exists Employee (Id int, Salary int)
Truncate table Employee
insert into Employee (id, salary) values ('1', '100')
insert into Employee (id, salary) values ('2', '200')
insert into Employee (id, salary) values ('3', '300')
"""

engine = run_sql_script(sql_script)

In [8]:
sql = """
SELECT MAX(salary) AS SecondHighestSalary
FROM Employee
WHERE salary <> (
    SELECT MAX(salary) FROM Employee
);
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)

df

,SecondHighestSalary
0,200


In [9]:
data = [[1, 100], [2, 200], [3, 300]]
employee = pd.DataFrame(data, columns=['id', 'salary']).astype({'id': 'int64', 'salary':'int64'})

In [10]:
max_salary = employee["salary"].max()
filtered = employee.loc[employee["salary"] != max_salary, "salary"]
pd.DataFrame({"SecondHighestSalary": [filtered.max()]})

,SecondHighestSalary
0,200


In [11]:
ser = employee["salary"].sort_values(ascending=False).drop_duplicates()
pd.DataFrame({"SecondHighestSalary": [None if ser.size < 2 else ser.iloc[1]]})

,SecondHighestSalary
0,200


### 177. Nth Highest Salary

In [12]:
def nth_highest_salary(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    ser = employee["salary"].sort_values(ascending=False).drop_duplicates()
    return pd.DataFrame(
        {f"getNthHighestSalary({N})": [None if N <= 0 or len(ser) < N else ser.iloc[N - 1]]}
    )

In [13]:
data = [[1, 100], [2, 200], [3, 300]]
employee = pd.DataFrame(data, columns=['id', 'salary']).astype({'id': 'int64', 'salary':'int64'})
nth_highest_salary(employee, 2)

,getNthHighestSalary(2)
0,200


In [14]:
data = [[1, 100]]
employee = pd.DataFrame(data, columns=['id', 'salary']).astype({'id': 'int64', 'salary':'int64'})
nth_highest_salary(employee, 2)

,getNthHighestSalary(2)
0,None


### 178. Rank Scores

In [15]:
sql_script = """
Create table If Not Exists Scores (id int, score DECIMAL(3,2))
Truncate table Scores
insert into Scores (id, score) values ('1', '3.5')
insert into Scores (id, score) values ('2', '3.65')
insert into Scores (id, score) values ('3', '4.0')
insert into Scores (id, score) values ('4', '3.85')
insert into Scores (id, score) values ('5', '4.0')
insert into Scores (id, score) values ('6', '3.65')
"""

engine = run_sql_script(sql_script)

In [16]:
sql = """
SELECT score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS rank
FROM Scores;
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)

df

,score,rank
0,4.00,1
1,4.00,1
2,3.85,2
3,3.65,3
4,3.65,3
5,3.50,4


In [17]:
data = [[1, 3.5], [2, 3.65], [3, 4.0], [4, 3.85], [5, 4.0], [6, 3.65]]
scores = pd.DataFrame(data, columns=['id', 'score']).astype({'id': 'Int64', 'score':'Float64'})

In [18]:
scores['rank'] = scores['score'].rank(method='dense', ascending=False)
scores.sort_values(by='rank').drop('id', axis=1)

,score,rank
2,4.0,1
4,4.0,1
3,3.85,2
1,3.65,3
5,3.65,3
0,3.5,4


### 180. Consecutive Numbers

In [19]:
sql_script = """
Create table If Not Exists Logs (id int, num int)
Truncate table Logs
insert into Logs (id, num) values ('1', '1')
insert into Logs (id, num) values ('2', '1')
insert into Logs (id, num) values ('3', '1')
insert into Logs (id, num) values ('4', '2')
insert into Logs (id, num) values ('5', '1')
insert into Logs (id, num) values ('6', '2')
insert into Logs (id, num) values ('7', '2')
"""

engine = run_sql_script(sql_script)

In [20]:
sql = """
SELECT DISTINCT num
FROM (
    SELECT num,
        LEAD(num, 1) OVER (ORDER BY id) AS next_num,
        LEAD(num, 2) OVER (ORDER BY id) AS next_num2
    FROM Logs
)
WHERE num = next_num AND next_num = next_num2
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,num
0,1


In [21]:
sql = """
SELECT DISTINCT l1.num AS ConsecutiveNums
FROM Logs l1, Logs l2, Logs l3
WHERE l1.id = l2.id - 1 AND l2.id = l3.id - 1
  AND l1.num = l2.num AND l2.num = l3.num
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,ConsecutiveNums
0,1


In [22]:
data = [[1, 1], [2, 1], [3, 1], [4, 2], [5, 1], [6, 2], [7, 2]]
logs = pd.DataFrame(data, columns=['id', 'num']).astype({'id': 'Int64', 'num':'Int64'})

In [23]:
logs[((logs['num'] == logs['num'].shift(-1)) &
      (logs['num'] == logs['num'].shift(-2)))]\
    .iloc[:, [1]].drop_duplicates()\
    .rename(columns={'num': 'ConsecutiveNums'})

,ConsecutiveNums
0,1


### 181. Employees Earning More Than Their Managers

In [24]:
sql_script = """
Create table If Not Exists Employee (id int, name varchar(255), salary int, managerId int)
Truncate table Employee
insert into Employee (id, name, salary, managerId) values ('1', 'Joe', '70000', '3')
insert into Employee (id, name, salary, managerId) values ('2', 'Henry', '80000', '4')
insert into Employee (id, name, salary, managerId) values ('3', 'Sam', '60000', NULL)
insert into Employee (id, name, salary, managerId) values ('4', 'Max', '90000', NULL)
"""

engine = run_sql_script(sql_script)

In [25]:
sql = """
SELECT e.name AS Employee
FROM Employee e
JOIN Employee e2 ON e.managerId = e2.id
WHERE e.salary > e2.salary;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Employee
0,Joe


In [26]:
sql = """
SELECT e1.name AS Employee
FROM Employee e1, Employee e2
WHERE e1.managerId = e2.id
  AND e1.salary > e2.salary;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Employee
0,Joe


### 182. Duplicate Emails

In [27]:
sql_script = """
Create table If Not Exists Person (id int, email varchar(255))
Truncate table Person
insert into Person (id, email) values ('1', 'a@b.com')
insert into Person (id, email) values ('2', 'c@d.com')
insert into Person (id, email) values ('3', 'a@b.com')
"""

engine = run_sql_script(sql_script)

In [28]:
sql = """
SELECT email Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Email
0,a@b.com


In [29]:
data = [[1, 'a@b.com'], [2, 'a@c.com'], [3, 'a@b.com']]
person = pd.DataFrame(data, columns=['id', 'email']).astype({'id': 'Int64', 'email':'object'})

In [30]:
person.groupby('email')[["id"]].count()\
    .query("id > 1")\
    .reset_index().rename(columns={"email": "Email"}).drop('id', axis=1)

,Email
0,a@b.com


In [31]:
person.groupby('email')["id"].count() \
    .loc[lambda x: x > 1] \
    .index.to_frame(name='Email').reset_index(drop=True)

,Email
0,a@b.com


In [32]:
person.groupby('email').size() \
    .loc[lambda x: x > 1] \
    .index.to_frame(name='Email').reset_index(drop=True)

,Email
0,a@b.com


In [33]:
person.groupby('email').size() \
    .loc[lambda x: x > 1] \
    .reset_index()[['email']].rename(columns={"email": "Email"})

,Email
0,a@b.com


In [34]:
person.groupby("email") \
    .filter(lambda x: len(x) > 1)[["email"]] \
    .drop_duplicates().rename(columns={"email": "Email"})

,Email
0,a@b.com


In [35]:
bool_ser = person['email'].duplicated()
person.loc[bool_ser, ["email"]] \
    .drop_duplicates().rename(columns={"email": "Email"})

,Email
2,a@b.com


### 183. Customers Who Never Order

In [36]:
sql_script = """
Create table If Not Exists Customers (id int, name varchar(255))
Create table If Not Exists Orders (id int, customerId int)
Truncate table Customers
insert into Customers (id, name) values ('1', 'Joe')
insert into Customers (id, name) values ('2', 'Henry')
insert into Customers (id, name) values ('3', 'Sam')
insert into Customers (id, name) values ('4', 'Max')
Truncate table Orders
insert into Orders (id, customerId) values ('1', '3')
insert into Orders (id, customerId) values ('2', '1')
"""

engine = run_sql_script(sql_script)

In [37]:
sql = """
SELECT name AS Customers
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
WHERE customerId IS NULL;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Customers
0,Henry
1,Max


In [38]:
sql = """
SELECT name AS Customers
FROM Customers 
WHERE id NOT IN (SELECT customerID FROM Orders);
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Customers
0,Henry
1,Max


In [39]:
data = [[1, 'Joe'], [2, 'Henry'], [3, 'Sam'], [4, 'Max']]
customers = pd.DataFrame(data, columns=['id', 'name']).astype({'id':'Int64', 'name':'object'})
data = [[1, 3], [2, 1]]
orders = pd.DataFrame(data, columns=['id', 'customerId']).astype({'id':'Int64', 'customerId':'Int64'})

In [40]:
orders.merge(customers, left_on='customerId', right_on='id', how='outer')\
    .query('customerId.isna()')[['name']].rename(columns={'name': 'Customers'})

,Customers
1,Henry
3,Max


In [41]:
df = customers.merge(orders, left_on="id", right_on="customerId", how="left") \
    .query("customerId.isna()")
df[["name"]].rename(columns={"name": "Customers"})

,Customers
1,Henry
3,Max


In [ ]:
df = customers[~customers['id'].isin(orders['customerId'])]
df[['name']].rename(columns={'name': 'Customers'})

,Customers
1,Henry
3,Max


: 

: 

### 184. Department Highest Salary

In [42]:
sql_script = """
Create table If Not Exists Employee (id int, name varchar(255), salary int, departmentId int)
Create table If Not Exists Department (id int, name varchar(255))
Truncate table Employee
insert into Employee (id, name, salary, departmentId) values ('1', 'Joe', '70000', '1')
insert into Employee (id, name, salary, departmentId) values ('2', 'Jim', '90000', '1')
insert into Employee (id, name, salary, departmentId) values ('3', 'Henry', '80000', '2')
insert into Employee (id, name, salary, departmentId) values ('4', 'Sam', '60000', '2')
insert into Employee (id, name, salary, departmentId) values ('5', 'Max', '90000', '1')
Truncate table Department
insert into Department (id, name) values ('1', 'IT')
insert into Department (id, name) values ('2', 'Sales')
"""

engine = run_sql_script(sql_script)

In [43]:
sql = """
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM (
    SELECT departmentId, MAX(salary) AS max_salary
    FROM Employee
    GROUP BY departmentId
) m
JOIN Department d ON m.departmentId = d.id
JOIN Employee e ON m.departmentId = e.departmentId AND m.max_salary = e.salary
;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Department,Employee,Salary
0,IT,Jim,90000
1,IT,Max,90000
2,Sales,Henry,80000


In [44]:
sql = """
SELECT d.name Department, e.name Employee, e.salary Salary
FROM Employee e
LEFT JOIN Department d ON e.departmentId = d.id
WHERE (e.departmentId, e.salary) IN (
    SELECT departmentId, MAX(salary)
    FROM Employee
    GROUP BY departmentId
);
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Department,Employee,Salary
0,IT,Jim,90000
1,Sales,Henry,80000
2,IT,Max,90000


In [45]:
sql = """
SELECT d.name Department, e.name Employee, e.salary Salary
FROM (
    SELECT *,
        DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS rnk 
    FROM Employee
) e
LEFT JOIN Department d ON e.departmentId = d.id
WHERE rnk = 1
;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Department,Employee,Salary
0,IT,Jim,90000
1,IT,Max,90000
2,Sales,Henry,80000


In [46]:
data = [[1, 'Joe', 70000, 1], [2, 'Jim', 90000, 1], [3, 'Henry', 80000, 2], [4, 'Sam', 60000, 2], [5, 'Max', 90000, 1]]
employee = pd.DataFrame(data, columns=['id', 'name', 'salary', 'departmentId'])\
    .astype({'id':'Int64', 'name':'object', 'salary':'Int64', 'departmentId':'Int64'})
data = [[1, 'IT'], [2, 'Sales']]
department = pd.DataFrame(data, columns=['id', 'name']).astype({'id':'Int64', 'name':'object'})

In [47]:
df = employee.groupby("departmentId")["salary"].max().reset_index() \
    .merge(department, left_on="departmentId", right_on="id") \
    .merge(employee, on=["departmentId", "salary"]) \
        [['name_x', 'name_y', 'salary']]
df.columns = ['Department', 'Employee', 'Salary']
df

,Department,Employee,Salary
0,IT,Jim,90000
1,IT,Max,90000
2,Sales,Henry,80000


In [48]:
df = employee.merge(department, left_on='departmentId', right_on='id')
df = df.groupby('departmentId').apply(lambda x: x[x['salary'] == max(x['salary'])])\
    .reset_index(drop=True)[['name_y', 'name_x', 'salary']]
df.columns = ['Department', 'Employee', 'Salary']
df

,Department,Employee,Salary
0,IT,Jim,90000
1,IT,Max,90000
2,Sales,Henry,80000


In [49]:
employee['rnk'] = employee.groupby('departmentId')['salary'].rank(
    method='dense', ascending=False)
df = employee.merge(department, left_on='departmentId', right_on='id')\
    .query('rnk == 1')[['name_y', 'name_x', 'salary']]
df.columns = ['Department', 'Employee', 'Salary']
df

,Department,Employee,Salary
1,IT,Jim,90000
2,Sales,Henry,80000
4,IT,Max,90000


### 185. Department Top Three Salaries

In [83]:
sql_script = """
Create table If Not Exists Employee (id int, name varchar(255), salary int, departmentId int)
Create table If Not Exists Department (id int, name varchar(255))
Truncate table Employee
insert into Employee (id, name, salary, departmentId) values ('1', 'Joe', '85000', '1')
insert into Employee (id, name, salary, departmentId) values ('2', 'Henry', '80000', '2')
insert into Employee (id, name, salary, departmentId) values ('3', 'Sam', '60000', '2')
insert into Employee (id, name, salary, departmentId) values ('4', 'Max', '90000', '1')
insert into Employee (id, name, salary, departmentId) values ('5', 'Janet', '69000', '1')
insert into Employee (id, name, salary, departmentId) values ('6', 'Randy', '85000', '1')
insert into Employee (id, name, salary, departmentId) values ('7', 'Will', '70000', '1')
Truncate table Department
insert into Department (id, name) values ('1', 'IT')
insert into Department (id, name) values ('2', 'Sales')
"""

engine = run_sql_script(sql_script)

In [51]:
sql = """
SELECT d.name Department, e.name Employee, e.salary Salary
FROM (
    SELECT *,
        DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS rnk 
    FROM Employee
) e
LEFT JOIN Department d ON e.departmentId = d.id
WHERE rnk <= 3
;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Department,Employee,Salary
0,IT,Max,90000
1,IT,Joe,85000
2,IT,Randy,85000
3,IT,Will,70000
4,Sales,Henry,80000
5,Sales,Sam,60000


In [ ]:
sql = """
SELECT Department, Employee, Salary
FROM (
    SELECT d.name AS Department,
           e.name AS Employee,
           e.salary AS Salary,
           DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY e.salary DESC) AS r
    FROM Employee e, Department d
    WHERE e.departmentId = d.id
) m
WHERE r <= 3;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Department,Employee,Salary
0,IT,Max,90000
1,IT,Joe,85000
2,IT,Randy,85000
3,IT,Will,70000
4,Sales,Henry,80000
5,Sales,Sam,60000


In [52]:
data = [[1, 'Joe', 85000, 1], [2, 'Henry', 80000, 2], [3, 'Sam', 60000, 2], [
    4, 'Max', 90000, 1], [5, 'Janet', 69000, 1], [6, 'Randy', 85000, 1], [7, 'Will', 70000, 1]]
employee = pd.DataFrame(data, columns=['id', 'name', 'salary', 'departmentId']).astype(
    {'id': 'Int64', 'name': 'object', 'salary': 'Int64', 'departmentId': 'Int64'})
data = [[1, 'IT'], [2, 'Sales']]
department = pd.DataFrame(data, columns=['id', 'name']).astype({'id': 'Int64', 'name':'object'})

In [91]:
employee['rnk'] = employee.groupby('departmentId')['salary'].rank(
    ascending=False, method='dense')
df = employee.query('rnk <= 3')\
    .merge(department, left_on='departmentId', right_on='id')\
        [['name_y', 'name_x', 'salary']]
df.columns = ['Department', 'Employee', 'Salary']
df

,Department,Employee,Salary
0,IT,Joe,85000
1,Sales,Henry,80000
2,Sales,Sam,60000
3,IT,Max,90000
4,IT,Randy,85000
5,IT,Will,70000


### 196. Delete Duplicate Emails

In [54]:
data = [[1, 'john@example.com'], [2, 'bob@example.com'], [3, 'john@example.com']]
person = pd.DataFrame(data, columns=['id', 'email']).astype({'id': 'int64', 'email':'object'})

In [55]:
person.sort_values(by='id').drop_duplicates(subset='email', keep='first')

,id,email
0,1,john@example.com
1,2,bob@example.com


In [56]:
sql_script = """
Create table If Not Exists Person (Id int, Email varchar(255))
Truncate table Person
insert into Person (id, email) values ('1', 'john@example.com')
insert into Person (id, email) values ('2', 'bob@example.com')
insert into Person (id, email) values ('3', 'john@example.com')
"""

engine = run_sql_script(sql_script)

In [57]:
sql = """
DELETE FROM
Person WHERE id IN (
    SELECT id
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY Email ORDER BY Id) AS rn
        FROM Person
    ) t
    WHERE rn > 1
)
"""
with engine.connect() as conn:
    conn.execute(text(sql))
    df = pd.read_sql(text("""SELECT * FROM Person"""), conn)
df

,Id,Email
0,1,john@example.com
1,2,bob@example.com


### 197. Rising Temperature

In [58]:
sql_script = """
Create table If Not Exists Weather (id int, recordDate date, temperature int)
Truncate table Weather
insert into Weather (id, recordDate, temperature) values ('1', '2015-01-01', '10')
insert into Weather (id, recordDate, temperature) values ('2', '2015-01-02', '25')
insert into Weather (id, recordDate, temperature) values ('3', '2015-01-03', '20')
insert into Weather (id, recordDate, temperature) values ('4', '2015-01-04', '30')
"""

engine = run_sql_script(sql_script)

In [59]:
sql = """
SELECT w2.id
FROM Weather w1, Weather w2
WHERE julianday(w2.recordDate) - julianday(w1.recordDate) = 1
  AND w2.temperature > w1.temperature;
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,id
0,2
1,4


In [60]:
data = [[1, '2015-01-01', 10], [2, '2015-01-02', 25],
        [3, '2015-01-03', 20], [4, '2015-01-04', 30]]
weather = pd.DataFrame(data, columns=['id', 'recordDate', 'temperature']).astype({'id': 'Int64', 'recordDate':'datetime64[ns]', 'temperature':'Int64'})

In [61]:
data = [[1, '2000-12-16', 3], [2, '2000-12-15', -1]]
weather = pd.DataFrame(data, columns=['id', 'recordDate', 'temperature']).astype({'id': 'Int64', 'recordDate':'datetime64[ns]', 'temperature':'Int64'})

In [62]:
weather.sort_values(by='recordDate', inplace=True)
weather.loc[weather.diff(1).query(
    'recordDate == "1 days" and temperature > 0').index][['id']]

,id


### 262. Trips and Users

In [63]:
sql_script = """
Create table If Not Exists Trips (id int, client_id int, driver_id int, city_id int, status varchar(50), request_at varchar(50))
Create table If Not Exists Users (users_id int, banned varchar(50), role varchar(50))
Truncate table Trips
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('1', '1', '10', '1', 'completed', '2013-10-01')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('2', '2', '11', '1', 'cancelled_by_driver', '2013-10-01')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('3', '3', '12', '6', 'completed', '2013-10-01')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('4', '4', '13', '6', 'cancelled_by_client', '2013-10-01')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('5', '1', '10', '1', 'completed', '2013-10-02')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('6', '2', '11', '6', 'completed', '2013-10-02')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('7', '3', '12', '6', 'completed', '2013-10-02')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('8', '2', '12', '12', 'completed', '2013-10-03')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('9', '3', '10', '12', 'completed', '2013-10-03')
insert into Trips (id, client_id, driver_id, city_id, status, request_at) values ('10', '4', '13', '12', 'cancelled_by_driver', '2013-10-03')
Truncate table Users
insert into Users (users_id, banned, role) values ('1', 'No', 'client')
insert into Users (users_id, banned, role) values ('2', 'Yes', 'client')
insert into Users (users_id, banned, role) values ('3', 'No', 'client')
insert into Users (users_id, banned, role) values ('4', 'No', 'client')
insert into Users (users_id, banned, role) values ('10', 'No', 'driver')
insert into Users (users_id, banned, role) values ('11', 'No', 'driver')
insert into Users (users_id, banned, role) values ('12', 'No', 'driver')
insert into Users (users_id, banned, role) values ('13', 'No', 'driver')
"""

engine = run_sql_script(sql_script)

In [64]:
sql = """
SELECT request_at AS Day, ROUND(SUM(is_cancel) / COUNT(*), 2) AS `Cancellation Rate`
FROM (
    SELECT t.*, CASE 
            WHEN status = 'completed' THEN 0
            ELSE 1
        END AS is_cancel
    FROM Trips t
    JOIN Users u ON t.client_id = u.users_id
    JOIN Users u2 ON t.driver_id = u2.users_id
    WHERE u.banned = 'No' AND u2.banned = 'No'
    ) r
GROUP BY request_at
;
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Day,Cancellation Rate
0,2013-10-01,0.0
1,2013-10-02,0.0
2,2013-10-03,0.0


In [65]:
sql = """
SELECT t.request_at AS Day,
       ROUND(
           SUM(CASE WHEN t.status = 'completed' THEN 0 ELSE 1 END) / COUNT(*),
           2
       ) AS `Cancellation Rate`
FROM Trips t
JOIN Users u ON t.client_id = u.users_id
JOIN Users u2 ON t.driver_id = u2.users_id
WHERE u.banned = 'No'
  AND u2.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at;
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,Day,Cancellation Rate
0,2013-10-01,0.0
1,2013-10-02,0.0
2,2013-10-03,0.0


In [66]:
data = [['1', '1', '10', '1', 'completed', '2013-10-01'], ['2', '2', '11', '1', 'cancelled_by_driver', '2013-10-01'], ['3', '3', '12', '6', 'completed', '2013-10-01'], ['4', '4', '13', '6', 'cancelled_by_client', '2013-10-01'], ['5', '1', '10', '1', 'completed', '2013-10-02'],
        ['6', '2', '11', '6', 'completed', '2013-10-02'], ['7', '3', '12', '6', 'completed', '2013-10-02'], ['8', '2', '12', '12', 'completed', '2013-10-03'], ['9', '3', '10', '12', 'completed', '2013-10-03'], ['10', '4', '13', '12', 'cancelled_by_driver', '2013-10-03']]
trips = pd.DataFrame(data, columns=['id', 'client_id', 'driver_id', 'city_id', 'status', 'request_at']).astype(
    {'id': 'Int64', 'client_id': 'Int64', 'driver_id': 'Int64', 'city_id': 'Int64', 'status': 'object', 'request_at': 'object'})

data = [['1', 'No', 'client'], ['2', 'Yes', 'client'], ['3', 'No', 'client'], ['4', 'No', 'client'], [
    '10', 'No', 'driver'], ['11', 'No', 'driver'], ['12', 'No', 'driver'], ['13', 'No', 'driver']]
users = pd.DataFrame(data, columns=['users_id', 'banned', 'role']).astype({'users_id': 'Int64', 'banned':'object', 'role':'object'})

In [67]:
df = trips.merge(users, left_on='client_id', right_on='users_id')\
    .merge(users, left_on='driver_id', right_on='users_id')\
    .query("banned_x == 'No' and banned_y == 'No' and \
            request_at >= '2013-10-01' and request_at <= '2013-10-03'")\
    .groupby('request_at').agg(is_cancel=('status', lambda x: x[x != "completed"].size / x.size))\
    .reset_index()
df.columns = ['Day', 'Cancellation Rate']
df

,Day,Cancellation Rate
0,2013-10-01,0.333333
1,2013-10-02,0.000000
2,2013-10-03,0.500000


In [68]:
trips.merge(users, left_on='client_id', right_on='users_id').query("request_at >= '2013-10-01' and request_at <= '2013-10-03'")

,id,client_id,driver_id,city_id,status,request_at,users_id,banned,role
0,1,1,10,1,completed,2013-10-01,1,No,client
1,2,2,11,1,cancelled_by_driver,2013-10-01,2,Yes,client
2,3,3,12,6,completed,2013-10-01,3,No,client
3,4,4,13,6,cancelled_by_client,2013-10-01,4,No,client
4,5,1,10,1,completed,2013-10-02,1,No,client
5,6,2,11,6,completed,2013-10-02,2,Yes,client
6,7,3,12,6,completed,2013-10-02,3,No,client
7,8,2,12,12,completed,2013-10-03,2,Yes,client
8,9,3,10,12,completed,2013-10-03,3,No,client
9,10,4,13,12,cancelled_by_driver,2013-10-03,4,No,client


In [ ]:
users

,users_id,banned,role
0,1,No,client
1,2,Yes,client
2,3,No,client
3,4,No,client
4,10,No,driver
5,11,No,driver
6,12,No,driver
7,13,No,driver


: 

: 

### 584. Find Customer Referee

In [69]:
sql_script = """
Create table If Not Exists Customer (id int, name varchar(25), referee_id int)
Truncate table Customer
insert into Customer (id, name, referee_id) values ('1', 'Will', NULL)
insert into Customer (id, name, referee_id) values ('2', 'Jane', NULL)
insert into Customer (id, name, referee_id) values ('3', 'Alex', '2')
insert into Customer (id, name, referee_id) values ('4', 'Bill', NULL)
insert into Customer (id, name, referee_id) values ('5', 'Zack', '1')
insert into Customer (id, name, referee_id) values ('6', 'Mark', '2')
"""

engine = run_sql_script(sql_script)

In [70]:
sql = """
SELECT name
FROM Customer
WHERE referee_id <> 2 OR referee_id IS NULL
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,name
0,Will
1,Jane
2,Bill
3,Zack


In [71]:
data = [[1, 'Will', None], [2, 'Jane', None], [3, 'Alex', 2],
        [4, 'Bill', None], [5, 'Zack', 1], [6, 'Mark', 2]]
customer = pd.DataFrame(data, columns=['id', 'name', 'referee_id']).astype({'id': 'Int64', 'name':'object', 'referee_id':'Int64'})

In [72]:
customer.query('(referee_id != 2) or (referee_id .isna())')[['name']]

,name
0,Will
1,Jane
3,Bill
4,Zack


In [73]:
customer[(customer['referee_id'] != 2) | (customer['referee_id'].isna())][['name']]

,name
0,Will
1,Jane
3,Bill
4,Zack


### 586. Customer Placing the Largest Number of Orders

In [74]:
sql_script = """
Create table If Not Exists orders (order_number int, customer_number int)
Truncate table orders
insert into orders (order_number, customer_number) values ('1', '1')
insert into orders (order_number, customer_number) values ('2', '2')
insert into orders (order_number, customer_number) values ('3', '3')
insert into orders (order_number, customer_number) values ('4', '3')
"""

engine = run_sql_script(sql_script)

In [75]:
sql = """
SELECT customer_number
FROM orders
GROUP BY customer_number
ORDER BY count(order_number) DESC LIMIT 1
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,customer_number
0,3


In [76]:
data = [[1, 1], [2, 2], [3, 3], [4, 3]]
orders = pd.DataFrame(data, columns=['order_number', 'customer_number']).astype({'order_number': 'Int64', 'customer_number':'Int64'})

In [77]:
pd.DataFrame({"customer_number": [orders.groupby('customer_number')['order_number'].count()
                                  .reset_index().sort_values(by='order_number', ascending=False).iloc[0].customer_number]})

,customer_number
0,3


In [78]:
orders['counts'] = orders.groupby('customer_number').transform('size')
orders.sort_values(by='counts', ascending=False).iloc[[0],[1]]

,customer_number
3,3


In [79]:
orders['customer_number'].mode().to_frame()

,customer_number
0,3


### 601. Human Traffic of Stadium

In [80]:
sql_script = """
Create table If Not Exists Stadium (id int, visit_date DATE NULL, people int)
Truncate table Stadium
insert into Stadium (id, visit_date, people) values ('1', '2017-01-01', '10')
insert into Stadium (id, visit_date, people) values ('2', '2017-01-02', '109')
insert into Stadium (id, visit_date, people) values ('3', '2017-01-03', '150')
insert into Stadium (id, visit_date, people) values ('4', '2017-01-04', '99')
insert into Stadium (id, visit_date, people) values ('5', '2017-01-05', '145')
insert into Stadium (id, visit_date, people) values ('6', '2017-01-06', '1455')
insert into Stadium (id, visit_date, people) values ('7', '2017-01-07', '199')
insert into Stadium (id, visit_date, people) values ('8', '2017-01-09', '188')
"""

engine = run_sql_script(sql_script)

In [81]:
sql = """
SELECT *
FROM Stadium
"""
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)
df

,id,visit_date,people
0,1,2017-01-01,10
1,2,2017-01-02,109
2,3,2017-01-03,150
3,4,2017-01-04,99
4,5,2017-01-05,145
5,6,2017-01-06,1455
6,7,2017-01-07,199
7,8,2017-01-09,188
